In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

dbutils.widgets.text("bronze_catalog", "bronze_germantelekom")
dbutils.widgets.text("schema", "dev_customer")
dbutils.widgets.text("landing_path", '/Volumes/bronze_germantelekom/dev_customer/landing/customers_day1.csv')

bronze_catalog = dbutils.widgets.get("bronze_catalog")
schema         = dbutils.widgets.get("schema")
#landing_path   = dbutils.widgets.get("landing_path")
landing_path = dbutils.widgets.get("landing_path").strip().strip("'\"")

bronze_tbl   = f"{bronze_catalog}.{schema}.customer_data"
rejected_tbl = f"{bronze_catalog}.{schema}.customer_data_rejected"
file_name    = landing_path.rstrip("/").split("/")[-1]       # one file per run

print("Reading :", landing_path)
print("Writing :", bronze_tbl, "| rejected ->", rejected_tbl)

# ---------- NEW: pass the batch file name to the next task (silver) ----------
try:
    dbutils.jobs.taskValues.set(key="source_file_name", value=file_name)
    print("Task value set: source_file_name =", file_name)
except Exception as e:
    print("Not running in a job, task value not set:", e)

# COMMAND ----------

# 1. Skip the file if it was already loaded (bronze or rejected)
already = (spark.table(bronze_tbl).filter(F.col("source_file_name") == file_name).limit(1).count()
           + spark.table(rejected_tbl).filter(F.col("source_file_name") == file_name).limit(1).count())
if already > 0:
    dbutils.notebook.exit(f"{file_name} is already loaded")

# COMMAND ----------

# 2. Header check on the file itself
EXPECTED_COLS = ["customer_id", "customer_name", "pincode",
                 "installed_product", "speed", "segment"]

def normalize_columns(name):
    return (name.replace("\ufeff", "")                       # hidden BOM at the start of a file
                .strip().lower().replace("_", " ").replace("-", " ").replace(".", ""))

actual_norm   = [normalize_columns(c) for c in spark.read.option("header", "true").csv(landing_path).columns]
expected_norm = [normalize_columns(c) for c in EXPECTED_COLS]

missing    = sorted(set(expected_norm) - set(actual_norm))
unexpected = sorted(set(actual_norm) - set(expected_norm))

if missing:
    raise Exception(f"Missing columns: {missing}")
elif unexpected:
    raise Exception(f"Unexpected columns: {unexpected}")
elif actual_norm != expected_norm:
    raise Exception(f"Column order changed. Expected {expected_norm}, found {actual_norm}")
print("Header check passed")

# COMMAND ----------

# 3. Read the data: all strings, malformed lines go to _corrupt_record
file_schema = StructType([
    StructField("customer_id",       StringType()),
    StructField("customer_name",     StringType()),
    StructField("pincode",           StringType()),
    StructField("installed_product", StringType()),
    StructField("speed",             StringType()),
    StructField("segment",           StringType()),
    StructField("_corrupt_record",   StringType()),
])

raw = (spark.read.format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("encoding", "UTF-8")
    .option("quote", '"')
    .option("escape", '"')
    .schema(file_schema)
    .load(landing_path)
    .withColumn("source_file_name", F.lit(file_name))
    .withColumn("load_timestamp", F.current_timestamp())
    .cache())

raw_n = raw.count()          # fills the cache before any filter on _corrupt_record

# COMMAND ----------

# 4. Split: clean rows -> bronze, malformed lines -> rejected
clean = raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

bad = (raw.filter(F.col("_corrupt_record").isNotNull())
          .select(F.col("_corrupt_record").alias("corrupt_record"),
                  F.lit("malformed record").alias("reject_reason"),
                  "source_file_name", "load_timestamp"))

clean_n, bad_n = clean.count(), bad.count()

# COMMAND ----------

# 5. Reconciliation BEFORE writing
src_rows = (spark.read.text(landing_path).filter(F.trim("value") != "").count()) - 1   # independent of the CSV parser

print(f"source={src_rows} | read={raw_n} | clean={clean_n} | rejected={bad_n}")

if src_rows != raw_n:
    raise Exception(f"Rows lost while reading: source {src_rows} != read {raw_n}")
if raw_n != clean_n + bad_n:
    raise Exception(f"Split error: {raw_n} != {clean_n} + {bad_n}")

# COMMAND ----------

# 6. Write
cols = ["customer_id", "customer_name", "pincode", "installed_product", "speed",
        "segment", "source_file_name", "load_timestamp"]

table_cols = [f.name for f in spark.table(bronze_tbl).schema.fields]
assert cols == table_cols, f"Column mismatch. Code: {cols} | Table: {table_cols}"

clean.select(*cols).write.format("delta").mode("append").saveAsTable(bronze_tbl)
bad.write.format("delta").mode("append").saveAsTable(rejected_tbl)

# COMMAND ----------

# 7. Reconciliation AFTER writing: source = bronze + rejected
in_bronze   = spark.table(bronze_tbl).filter(F.col("source_file_name") == file_name).count()
in_rejected = spark.table(rejected_tbl).filter(F.col("source_file_name") == file_name).count()

print(f"bronze={in_bronze} | rejected={in_rejected} | total={in_bronze + in_rejected} | source={src_rows}")

if in_bronze != clean_n or in_rejected != bad_n or (in_bronze + in_rejected) != src_rows:
    raise Exception(
        f"Post-write mismatch: bronze {in_bronze} (expected {clean_n}), "
        f"rejected {in_rejected} (expected {bad_n}), source {src_rows}")

print("Bronze load complete")